In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import confusion_matrix, classification_report, silhouette_score, accuracy_score, silhouette_samples
from sklearn.cluster import KMeans
from sklearn.decomposition import SparsePCA

import warnings
warnings.filterwarnings('ignore', category = FutureWarning)

# Sub-task 1: Unsupervised Learning 
### Using Python, apply k-means to partition the data set into three clusters. Plot a graph of the resulting clusters. Your (fictional) colleague claims to have done the above multiple times. They claim that in their experience, k-means always converges quickly on this dataset and that the graphs that they get after each run of the algorithm (with different random seeds) show very similar clusters. They say that this shows that the three clusters found with k-means are the best possible clusters within this dataset. Do you agree with your colleague? Justify your answer based on your graph, and on your understanding of the k-means algorithm.

In [3]:
df1 = pd.read_csv(r'cluster.csv', header = None, sep = ',')
df1.columns = ['Point1', 'Point2']
df1 = df1.astype('Float32')
df1.describe()

,Point1,Point2
count,500.0,500.0
mean,3.943291,-0.997142
std,2.395397,1.704309
min,-1.1647,-4.9663
25%,1.940225,-1.241425
50%,3.0418,-0.65717
75%,6.58395,-0.0282
max,7.5723,1.5068


In [ ]:
# K-Means is scale-sensitive as it uses Euclidean distance to compute. High range or variance features control the clustering. In our dataset:
# Point1: Range = (-1.1647, 7.5723), Variance = (2.395397)^2 | Point2: Range = (-4.9663, 1.5068), Variance = (1.704309)^2
# Due to the differences in ranges and variances, scaling is required so that there is an equal contribution from all features.

df = df1.copy()
df_scaled = (df - df.min()) / (df.max() - df.min())
df_scaled.describe()

In [ ]:
# Sometimes, by looking at the Scatter plot, we can gain a better understanding of the data and potentially number of the clusters.
# Especially for this dataset, which has 500 data points and two variables that can be easily plotted in a 2D plane. 
# However, if the clusters were spherical and overlapping, it would be more challenging to estimate the number of clusters visually.
plt.figure(figsize = (17, 7))
plt.scatter(df_scaled['Point1'], df_scaled['Point2'], c = 'blue')
plt.xlabel('Point1')
plt.ylabel('Point2')
plt.title('Scatter Plot of cluster data set')
plt.show()

In [ ]:
def clustering(df, n_clusters = 3):
    '''
    This function returns the centroids and labels and pass them for next function to plot the data points + centroids.
    ----------------------------------------------------------
    df: The DataFrame containing the data points.
    n_clusters: The number of clusters to create. Default is set to 3.
    '''
    kmeans = KMeans(n_clusters = n_clusters, random_state = 92)
    kmeans.fit(df)

    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
        
    return centroids, labels

In [ ]:
def cluster_visualization(df, centroids, labels, n_clusters = 3):
    '''
    This function visualizes the clustering results by plotting each cluster and its centroids.
    -------------------------------------------------------------------
    df: The DataFrame containing the data points.
    centroids: The centroids of the clusters.
    labels: The cluster labels for each data point.
    n_clusters: The number of clusters to visualize. Default is set to 3.
    '''
    
    plt.figure(figsize=(10, 5))

    for label in range(n_clusters):
        plt.scatter(df.loc[labels == label, 'Point1'], df.loc[labels == label, 'Point2'], label=f'Cluster {label + 1}') # plot datapoints for each cluster

    plt.scatter(centroids[:, 0], centroids[:, 1], c='grey', s=200, alpha=0.5, label='Centroids') # plot the centroids

    plt.xlabel('Point1')
    plt.ylabel('Point2')
    plt.title(f'{n_clusters} Clusters of data set')
    plt.legend()
    plt.show()

In [ ]:
silo_score_avg = []

for num in range(2, 11):  # for each number of cluster, take random state from 1 to 100 then average the silhouette score and plot them
    scores = []
    for rand in np.arange(1, 100):
        kmeans = KMeans(n_clusters = num, random_state = rand, n_init = num)
        kmeans.fit(df_scaled)
        scores.append(silhouette_score(df_scaled, kmeans.labels_))
    silo_score_avg.append(np.mean(scores))
        

silhouette_avg = sum(silo_score_avg) / len(silo_score_avg)

print("The overal average silhouette_score is:", np.round(silhouette_avg, 4))
for i, avg in enumerate(silo_score_avg): # We can comment this part, beacaues in the next steps we will be ploting it. 
    print(f"Cluster {i + 2}: Silhouette Score = {avg:.4f}") 

In [ ]:
elbow_score_avg = []  # List to store WCSS values
cluster_range = range(2, 11)  # Range of clusters and n_init set to range of (2 to 10)

for num in cluster_range:
    elbow_score = []
    for rand in np.arange(1, 100):
        kmeans = KMeans(n_clusters = num, random_state = rand, n_init = num)
        kmeans.fit(df1)
        elbow_score.append(kmeans.inertia_)
    elbow_score_avg.append(np.mean(elbow_score))

elbow_avg = np.mean(elbow_score_avg)

print("The overal average elbow_score is:", np.round(elbow_avg, 4)) # The elbow score keeps getting smaller as the number of clusters increases.
for i, avg in enumerate(elbow_score_avg): # Like previous part, we can comment this part, beacaues in the next step we will be ploting it. 
    print(f"Cluster {i + 2}: Elbow Score = {avg:.4f}")

# Ploting the results:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

ax[0].plot(cluster_range, silo_score_avg, marker='o')
ax[0].set_xlabel('Number of Clusters')
ax[0].set_ylabel('Silhouette Score avg')
ax[0].set_title('Silhouette Scores for Different Cluster Numbers')
ax[0].grid()

ax[1].plot(cluster_range, elbow_score_avg, marker='o', color='g')
ax[1].set_xlabel('Number of Clusters')
ax[1].set_ylabel('SSE')
ax[1].set_title('Elbow Method for Optimal Number of Clusters')
ax[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# This part of code is from sklearn documentation, however, some modification are made to fit data and adjust the plot.
range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    # Create a subplot with 1 row and 2 columns
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # The 1st subplot is the silhouette plot
    # The silhouette coefficient can range from -1, 1 but in this example all
    ax1.set_xlim([-0.3, 1])
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    ax1.set_ylim([0, len(df_scaled) + (n_clusters + 1) * 10])

    # Initialize the clusterer with n_clusters value and a random generator
    # seed of 10 for reproducibility.
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(df_scaled)
    

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed
    # clusters
    silhouette_avg = silhouette_score(df_scaled, cluster_labels)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(df_scaled, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title("The silhouette plot for the various clusters.")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

    ax1.set_yticks([])  # Clear the yaxis labels / ticks
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    # 2nd Plot showing the actual clusters formed
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        df_scaled.iloc[:, 0], df_scaled.iloc[:, 1], marker=".", s=30, lw=0, alpha=0.7, c=colors, edgecolor="k"
    )

    # Labeling the clusters
    centers = clusterer.cluster_centers_
    # Draw white circles at cluster centers
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    for i, c in enumerate(centers):
        ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

    ax2.set_title("The visualization of the clustered data.")
    ax2.set_xlabel("Feature space for the 1st feature")
    ax2.set_ylabel("Feature space for the 2nd feature")

    plt.suptitle(
        f"Silhouette analysis for KMeans clustering on sample data with n_clusters = {n_clusters}, silhouette score = {np.round(silhouette_avg, 4)}.",
        fontsize=14,
        fontweight="bold",
    )

plt.show()

In [ ]:
for k in [3, 4, 5]:
    print(f'For {k} clusters:')
    centroids, labels = clustering(df_scaled, n_clusters = k)
    cluster_visualization(df_scaled, centroids, labels, n_clusters = k)
    print(19 * '------')


# Clustering Analysis Summary

## Key Findings

- **Challenging the Claim:**  
  The statement that _"3 clusters are always best"_ is **not supported** by the analysis.

- **Random Seed Sensitivity:**  
  Even with `k-means++` initialization, clustering results **vary across different random seeds** (tested with 99 seeds per cluster count).

- **Optimal Number of Clusters:**  
  - **4 or 5 clusters** yield higher average silhouette scores and better-defined groupings compared to 3 clusters.
  - For **5 clusters**, some points in cluster 1 are not properly assigned, indicating possible overlaps or limitations in centroid placement.

## Computational Considerations

- **Robustness vs. Efficiency:**  
  Running K-means with multiple seeds (`n_init` from 2 to 10, 99 seeds each) gives a more robust view but is **computationally expensive** (~3.5 minutes per run on an i7 8th Gen laptop).

- **Alternative Algorithms:**  
  - **Density-based clustering (DBSCAN, HDBSCAN):** These methods do not rely on initial centroids or random seeds and may handle noise/outliers better.
  - **Hierarchical clustering:** Consider if the data has a meaningful hierarchy.

> **Note:** While `k-means++` improves initialization, randomness still affects results. Exploring alternative clustering methods can provide more stable and meaningful groupings for this dataset.
---

**Reference**:  
1- Arthur, D., & Vassilvitskii, S. (2007). *k-means++: The advantages of careful seeding*. [PDF Link](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf)

2- Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). *A density-based algorithm for discovering clusters in large spatial databases with noise*. [PDF Link](https://www.aaai.org/Papers/KDD/1996/KDD96-037.pdf)

3- Leland McInnes, John Healy, and Steve Astels hdbscan: Hierarchical density based clustering [Web page](https://www.researchgate.net/publication/315508524_hdbscan_Hierarchical_density_based_clustering)


---------------------------------------

# Sub-task 2: Dimensionality Reduction 

###  Use scikit learn’s implementation of sparse PCA to fit a model with two components to the raw data. Sparse PCA is a version of PCA which attempts to find sparse components. The idea is similar to that discussed for factor analysis. Sparsity is controlled by a parameter alpha. For this task, set alpha to 5. Print out the components found by sparse PCA. Now scale the data so that it has mean 0 and variance 1 (you may wish to use scikit learn’s Standard Scaler). Similarly to above, use sparse PCA to fit a model with two components to the scaled data and print out the resulting components. State whether you see a significant difference between the two results, explaining your observation with reference to the properties of the dataset and of PCA. If there is a significant difference between your results state which of the two results is most likely to capture the relationships between the variables. For the result that you think is most likely to capture these relationships, discuss whether or not it supports the potential latent variable model we discussed in lectures.

In [ ]:
df2 = pd.read_csv('cars_reduced.csv', header = None, sep = ',')
df2. columns = ['retail price', 'dealer price', 'wheel base', 'length']
df2 = df2.astype('Float32')
df2.head(3)

In [ ]:
# By looking at the box plot, we can see that there are some outliers in the data, which can affect the results of the SparsePCA algorithm.
plt.figure(figsize = (17, 7))
fig, ax = plt.subplots(2, 2, figsize=(15, 5))
fig.suptitle('Box plot of the data set')
ax[0, 0].boxplot(df2['retail price'], vert = False)
ax[0, 1].boxplot(df2['dealer price'], vert = False)
ax[1, 0].boxplot(df2['wheel base'], vert = False)
ax[1, 1].boxplot(df2['length'], vert = False)

ax[0, 0].set_title('retail price')
ax[0, 1].set_title('dealer price')
ax[1, 0].set_title('wheel base')
ax[1, 1].set_title('length')

ax[0, 0].grid()
ax[0, 1].grid()
ax[1, 0].grid()
ax[1, 1].grid()

plt.tight_layout()
plt.show()
print(round(df2.describe(), 2))

print(45 * '----')
# 2- With Scaling
ss = StandardScaler()
df2_scaled = ss.fit_transform(df2)
df2_scaled = pd.DataFrame(df2_scaled, columns = ['retail price', 'dealer price', 'wheel base', 'length'])

plt.figure(figsize = (17, 7))
fig, ax = plt.subplots(2, 2, figsize=(15, 5))
fig.suptitle('Box plot of the scaled data set')
ax[0, 0].boxplot(df2_scaled['retail price'], vert = False)
ax[0, 1].boxplot(df2_scaled['dealer price'], vert = False)
ax[1, 0].boxplot(df2_scaled['wheel base'], vert = False)
ax[1, 1].boxplot(df2_scaled['length'], vert = False)

ax[0, 0].set_title('retail price')
ax[0, 1].set_title('dealer price')
ax[1, 0].set_title('wheel base')
ax[1, 1].set_title('length')

ax[0, 0].grid()
ax[0, 1].grid()
ax[1, 0].grid()
ax[1, 1].grid()

plt.tight_layout()
plt.show()
print(round(df2_scaled.describe(), 2))


In [ ]:
# 1- Without scaling
pca_raw = SparsePCA(n_components = 2, alpha = 5, random_state = 71)
pca_raw.fit(df2)
df_pca = pca_raw.transform(df2)
df_pca = pd.DataFrame(df_pca, columns = ['PC1', 'PC2'])


# 2- With scaling
pca_scaled = SparsePCA(n_components = 2, alpha = 5, random_state = 71)
pca_scaled.fit(df2_scaled)
df_pca = pca_scaled.transform(df2_scaled)
df_pca = pd.DataFrame(df_pca, columns = ['PC1', 'PC2'])


print('Before scaling & doing SparsePCA:') # Show the results
print("Raw data components:\n", pca_raw.components_)
plt.bar(pca_raw.components_.T[0], pca_raw.components_.T[1])
plt.xlabel('Principal Components')
plt.ylabel('Feature Values')
plt.title('Raw data components')
plt.show()

print(8 * '----')
print('Scaling & SparsePCA:')
print("Scaled data components:\n", pca_scaled.components_)
plt.bar(pca_scaled.components_.T[0], pca_scaled.components_.T[1])
plt.xlabel('Principal Components')
plt.ylabel('Feature Values')
plt.title('Scaled data components')
plt.show()


PCA Sensitivity to Variance:
Raw data components are skewed toward high-variance features (price variables, which typically have larger numerical values than length measurements). This obscures relationships between variables. Effect of Standardization: Scaling ensures all features contribute equally to the variance. Sparse PCA can then isolate latent variables based on correlations, not scale differences.

The scaled data results are far more meaningful: They align perfectly with the proposed latent variable model: Component 1: Captures price aspects (retail_price + dealer_price). Component 2: Captures length aspects (wheel_base + length). The symmetry (weights ≈ 0.707) reflects equal contribution from standardized variables, a hallmark of PCA on scaled data. The raw data results are misleading due to scale-driven bias.


The clean separation into price and length components directly matches the lecture’s hypothesis of two latent variables. The near-identical weights (0.707 ≈ 1/√2) suggest: Price variables are equally influenced by the first latent factor. Length variables are equally influenced by the second latent factor. Raw data does not support the model due to mixed components.

**Conclusion**
Always scale data before PCA (including sparse PCA). The scaled results validate the latent variable model, while raw data obscures it due to scale differences. This demonstrates how preprocessing critically impacts the interpretability of dimensionality reduction techniques.

# Sub-task 3 
### 

In [4]:
df3 = pd.read_csv(r'stroke-data.csv', sep = ',')
df3.info() # bmi has some missing values. So, either: 1- remove them or 2- impute them with mean, median, etc. But, as instructed, all the missing values will be removed.
df3_clean = df3.copy()
df3_clean.dropna(axis = 0, inplace=True)
df3_clean.set_index('id', inplace = True)
df3_clean.drop(['work_type', 'Residence_type'], axis = 1, inplace = True) # No strong relationship with the target column.
print(17 * '----')
print(f'# rows before cleaning: {len(df3)} VS after cleaning: {len(df3_clean)}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
--------------------------------------------------------------------
# rows before cleaning: 5110 VS after cleaning: 4909


In [ ]:
plt.figure(figsize = (5, 7))
plt.pie(df3_clean['stroke'].value_counts(), explode = [0.1, 0.1], 
        labels = [f'No Stroke: {len(df3_clean[df3_clean.stroke == 0])}', f'Stroke: {len(df3_clean[df3_clean.stroke == 1])}'], 
        autopct = '%1.1f%%', startangle = 90)
plt.legend()
plt.title('Count of strokes')
plt.show() # Based on the plot we have an imbalanced dataset

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 5))
ax = ax.flatten()

for idx, column in enumerate(df3_clean[['age', 'avg_glucose_level', 'bmi']]): #.select_dtypes(include=np.number).columns
    sns.kdeplot(df3_clean[column], ax=ax[idx], fill=True)
    ax[idx].set_xlabel(column)
    ax[idx].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
# Converting categorical columns to numeric. Either replacing or one-hot encoding which I choose replacing in order to avoid increasing number of columns dimensionality.
df3_clean.gender.replace({'Male': 1, 'Female': 0, 'Other': 1}, inplace = True)

df3_clean.ever_married.replace({'Yes': 1, 'No': 0}, inplace = True)

df3_clean.smoking_status.replace({'Unknown': 0, 'never smoked': 1, 'formerly smoked': 2, 'smokes': 3}, inplace = True)

df3_clean['age_cat'] = pd.cut(df3_clean['age'], bins=[0, 50, 80, float('Inf')], labels=['young adult', 'middle-aged', 'very old']) # Accordng to studies in the additional files.
df3_clean = pd.get_dummies(df3_clean, columns = ['age_cat'], drop_first = False, prefix = '')
df3_clean[['_very old', '_middle-aged', '_young adult']] = df3_clean[['_very old', '_middle-aged', '_young adult']].astype(int)
df3_clean.columns = df3_clean.columns.str.replace('^_', '', regex=True)

df3_clean = df3_clean[['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'avg_glucose_level',
 'bmi', 'smoking_status', 'young adult', 'middle-aged', 'very old','stroke']] # Reordering columns for better visualization

In [ ]:
for i in  df3_clean.select_dtypes(include = np.number):
    print(df3_clean[i].value_counts()) # empty_dict[i] = df3_clean[i].value_counts() >>> create a dictionary instead of list
    print(8 * '----')

In [ ]:
plt.figure(figsize = (15, 10))
corr = (df3_clean.select_dtypes(include = np.number)).corr()
sns.heatmap(corr, annot = True, cmap = 'coolwarm', vmax = 1, vmin = -1)
plt.title('Correlation Matrix')
plt.xlabel('Features')
plt.ylabel('Features')
plt.show()
# According to the plot, there is no strong correlation between the features and most interesting the bmi and smoking status has very weak correlation with the target column

In [ ]:
df3_final = df3_clean.copy()
X, y = df3_final.drop('stroke', axis = 1), df3_final.stroke
X_train, X_dev, y_train, y_dev = train_test_split(X, y, train_size = 0.8, random_state = 71, stratify = y)
X_test, X_val, y_test, y_val = train_test_split(X_dev, y_dev, train_size = 0.5, random_state = 71, stratify = y_dev)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'Val size: {len(X_val)}')
print(f'label0 in train: {len(y_train[y_train == 0])}')
print(f'label1 in train: {len(y_train[y_train == 1])}')
print(f'label0 in test: {len(y_test[y_test == 0])}')
print(f'label1 in test: {len(y_test[y_test == 1])}')
print(f'label0 in val: {len(y_val[y_val == 0])}')
print(f'label1 in val: {len(y_val[y_val == 1])}')
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
X_val = sc.transform(X_val)

In [ ]:
# MLP, SVM, Random Forest, Decision Tree, KNN
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [ ]:
def model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'Result of {model}:')
    print(classification_report(y_test, y_pred))
    print(8 * '----')

In [ ]:
model(MLPClassifier(hidden_layer_sizes=(100, 50, 25),  solver = 'adam', learning_rate = 'adaptive', random_state = 71), X_train, y_train, X_val, y_val)
model(SVC(random_state=71, kernel = 'rbf'), X_train, y_train, X_val, y_val)
model(RandomForestClassifier(random_state=71, n_estimators = 100, max_depth = 5, min_samples_leaf = 2), X_train, y_train, X_val, y_val)
model(DecisionTreeClassifier(random_state=71, max_depth = 5, min_samples_leaf = 2), X_train, y_train, X_val, y_val)
model(KNeighborsClassifier(n_neighbors = 2), X_train, y_train, X_val, y_val)

### Now try to manipulate the data to get even number of target labels.

In [ ]:
df3_final2 = df3_clean.copy()

label1 = df3_final2[df3_final2.stroke == 1] # taking minority label out of the dataset
label0 = df3_final2[df3_final2.stroke == 0].sample(len(label1), random_state = 71) # taking majority label out of the dataset

df3_final2_1 = pd.concat([label1, label0], axis = 0)
df3_finalRemain = df3_final2[~df3_final2.index.isin(df3_final2_1.index)] # taking the rest of the dataset
Xr, yr = df3_finalRemain.drop('stroke', axis = 1), df3_finalRemain.stroke

X, y = df3_final2_1.drop('stroke', axis = 1), df3_final2_1.stroke
X_train, X_dev, y_train, y_dev = train_test_split(X, y, train_size = 0.8, random_state = 71)
X_test, X_val, y_test, y_val = train_test_split(X_dev, y_dev, train_size = 0.5, random_state = 71)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'Val size: {len(X_val)}')
print(f'label0 in train: {len(y_train[y_train == 0])}')
print(f'label1 in train: {len(y_train[y_train == 1])}')
print(f'label0 in test: {len(y_test[y_test == 0])}')
print(f'label1 in test: {len(y_test[y_test == 1])}')
print(f'label0 in val: {len(y_val[y_val == 0])}')
print(f'label1 in val: {len(y_val[y_val == 1])}')
print(8 * '----')
print(f'Remain Size: {len(df3_finalRemain)}')

In [ ]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)
X_val_sc = sc.transform(X_val)
xr_sc = sc.transform(Xr)

In [ ]:
model(MLPClassifier(hidden_layer_sizes=(100, 50, 25),  solver = 'adam', learning_rate = 'adaptive', random_state = 71, max_iter = 1500), X_train_sc, y_train, X_val_sc, y_val)
model(SVC(random_state=71, kernel = 'rbf'), X_train_sc, y_train, X_val_sc, y_val)
model(RandomForestClassifier(random_state=71, n_estimators = 100, max_depth = 5, min_samples_leaf = 2), X_train_sc, y_train, X_val_sc, y_val)
model(DecisionTreeClassifier(random_state=71, max_depth = 5, min_samples_leaf = 2), X_train_sc, y_train, X_val_sc, y_val)
model(KNeighborsClassifier(n_neighbors = 2), X_train_sc, y_train, X_val_sc, y_val)

In [ ]:
#model(SVC(random_state=71, kernel = 'rbf'), X_train_sc, y_train, X_val_sc, y_val)
svc = SVC(random_state=71, kernel = 'rbf')
svc.fit(X_train_sc, y_train)
y_pred = svc.predict(X_val_sc)
print(classification_report(y_val, y_pred))
print(8 * '----')
y_pred_xr = svc.predict(xr_sc)
print(classification_report(yr, y_pred_xr))